In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

# 05.3 - Deep Agents: Human-in-the-Loop Research Loop

This notebook introduces `create_deep_agent()` from the `deepagents` package - a higher-level API that wraps the `StateGraph` + `ToolNode` pattern you built in **05.2** and adds a permission layer on top.

## From StateGraph to `create_deep_agent()`

In 05.2, you wired a `StateGraph` manually. `create_deep_agent()` handles that wiring internally:

| 05.2 - manual `StateGraph` | 05.3 - `create_deep_agent()` |
|---|---|
| `StateGraph(MessagesState)` + `add_node` + `add_edge` + `compile()` | handled internally |
| `ToolNode(tools)` | `tools=[internet_search]` |
| No checkpointer | `checkpointer=MemorySaver()` |
| No HITL - agent runs to completion | `permissions=[FilesystemPermission(...)]` pauses before writes |
| - | Resume: `agent.invoke(Command(resume={...}), config)` |

The key insight: **oversight is a property of agent construction, not graph wiring.** You don't add a human approval node - you declare a permission rule at construction time, and the framework pauses automatically when the rule triggers.

## What we'll build

A research loop that:
1. Takes a topic from you
2. Searches the web via Tavily
3. **Pauses** before writing results to disk - showing you what it wants to save
4. Proceeds based on your approval or rejection
5. Assesses its own output quality regardless of what you decided

**Prerequisites:** Complete `05_2_langchain_agent.ipynb` before this notebook.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_tavily import TavilySearch
from deepagents import create_deep_agent, FilesystemPermission
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langchain_core.messages import AIMessage
from IPython.display import Markdown, display

llm = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.3,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

display(Markdown(f"LLM ready: {llm is not None}"))
display(Markdown(f"API_GATEWAY_KEY loaded: {bool(os.getenv('API_GATEWAY_KEY'))}"))
display(Markdown(f"TAVILY_API_KEY loaded: {bool(os.getenv('TAVILY_API_KEY'))}"))

## Section 1 - Tool Definition

`create_deep_agent()` accepts standard LangChain `@tool`-decorated functions - the same pattern from `tools_animals.py` and `tools_horoscope.py` in the course chat client. The agent infers the tool's schema from the function signature and docstring; no separate schema object is needed.

We wrap `TavilySearch` in a thin `@tool` function to keep the pattern consistent with the rest of the course.

In [ ]:
@tool
def internet_search(query: str, max_results: int = 5) -> str:
    """Search the web for information on a topic. Returns a summary of the top results."""
    results = TavilySearch(max_results=max_results).invoke({"query": query})
    return str(results)

In [ ]:
# Verify TAVILY_API_KEY is working before building the agent
test_result = internet_search.invoke({"query": "LangGraph human-in-the-loop 2024"})
print(test_result[:500])


## Section 2 - Agent Construction with FilesystemPermission

Three components work together to enable the permission gate:

1. **`FilesystemPermission(mode="interrupt")`** - fires *before any write hits disk*, including the agent's built-in `write_file` harness tool. The gate is filesystem-level, not tied to a specific tool name. `paths=["/**"]` means any path triggers it.

2. **`MemorySaver()`** - required for HITL. When the interrupt fires, the agent suspends mid-execution. `MemorySaver` preserves the full graph state so the `resume` call can reconstruct exactly where it stopped.

3. **`thread_id`** - the session key. The same `thread_id` must appear in both the initial `invoke()` and the `Command(resume=...)` call. Use a fresh `thread_id` for each independent demo run to avoid state collision between runs.

In [ ]:
permission = FilesystemPermission(
    operations=["write"],
    paths=["/**"],
    mode="interrupt",
)
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    permissions=[permission],
    checkpointer=checkpointer,
)

print("Agent constructed:", agent is not None)

In [ ]:
agent

## Section 3 - Research Loop: Approve Path

Run the agent with a research topic. It will search the web, then **pause** and ask for permission before writing results to disk.

The two-cell pattern below is the pedagogical center of this notebook:
- **Cell A - Invoke**: agent runs until it hits the write gate, execution suspends, `result.interrupts` is non-empty
- **Cell B - Resume**: you send a decision, agent continues from where it stopped

We guide the agent's save path with a filename hint in the prompt (OQ2 resolution).

In [ ]:
topic = "recent advances in retrieval-augmented generation"
config = {"configurable": {"thread_id": "research-1"}}

prompt = f"Research this topic and save a summary to rag_research.md: {topic}"
result = agent.invoke(
    {"messages": [("user", prompt)]},
    config=config,
    version="v2",
)

pending = len(result.interrupts) if result.interrupts else 0
print(f"Invoke complete. Interrupts pending: {pending}")

The agent stopped before writing. Inspect what it wants to do, then decide.

In [ ]:
# Inspect the pending write request
if result.interrupts:
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value.get("action_requests", [])
    if action_requests:
        req = action_requests[0]
        print("Agent wants to write:")
        print("  Path    :", req.get("path", "unknown"))
        content = str(req.get("content", ""))
        preview = content[:400] + ("..." if len(content) > 400 else "")
        print("  Preview :", preview)
    else:
        print("Interrupt payload:", interrupt_value)
else:
    print("Agent completed without requesting a write - no interrupt to inspect.")
    print("This can happen if the agent decided the results did not warrant saving.")
    print("Skip to Section 5 to run the self-assessment.")

In [ ]:
# Approve - agent resumes and writes the file
approved_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    version="v2",
)

# Normalize for Section 5 - last_result holds the most recent completed run
last_result = approved_result

# Show the agent's final AI message
msgs = last_result["messages"] if isinstance(last_result, dict) else last_result.value['messages']
final_msg = next(
    (m for m in reversed(msgs) if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)),
    None
)
if final_msg:
    print("Agent final message:")
    print(final_msg.content)

In [ ]:
# Verify the file was written
saved_path = "rag_research.md"
if os.path.exists(saved_path):
    print(f"File written: {saved_path} ({os.path.getsize(saved_path)} bytes)")
else:
    nearby_md = [f for f in os.listdir(".") if f.endswith(".md")]
    if nearby_md:
        print("Markdown files found nearby:", nearby_md)
    else:
        print("No markdown file found in current directory.")
        print("The agent may have written to a different path - check the content preview above.")

## Section 4 - Research Loop: Reject Path

The reject path uses a **fresh `thread_id`** so it starts with a clean state - no bleed from the approve run above.

When you reject, the agent:
- Skips the file write entirely (R6)
- Acknowledges your decision in its final message
- Does **not** retry - the loop ends after one rejection

In [ ]:
# Fresh thread_id - independent session, no state from Section 3
config_reject = {"configurable": {"thread_id": "research-2"}}

prompt_reject = f"Research this topic and save a summary to rag_research_reject.md: {topic}"
result_reject = agent.invoke(
    {"messages": [("user", prompt_reject)]},
    config=config_reject,
    version="v2",
)

pending_r = len(result_reject.interrupts) if result_reject.interrupts else 0
display(Markdown(f"Invoke complete. Interrupts pending: {pending_r}"))

In [ ]:
# Inspect the pending write (same pattern as Section 3)
if result_reject.interrupts:
    interrupt_value_r = result_reject.interrupts[0].value
    action_requests_r = interrupt_value_r.get("action_requests", [])
    if action_requests_r:
        print("Agent wants to write:", action_requests_r[0].get("path", "unknown"))
    print("\nDeciding to reject...")
else:
    print("No interrupt - agent completed without requesting a write.")

In [ ]:
# Reject - agent skips the write and acknowledges
rejected_result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Student declined the save."}]}),
    config=config_reject,
    version="v2",
)

# Normalize for Section 5 - overwrite last_result with the reject-path result
last_result = rejected_result

msgs_r = last_result["messages"] if isinstance(last_result, dict) else last_result.value['messages']
final_msg_r = next(
    (m for m in reversed(msgs_r) if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)),
    None
)
if final_msg_r:
    print("Agent acknowledged rejection:")
    print(final_msg_r.content)

In [ ]:
# Confirm no file was written
if os.path.exists("rag_research_reject.md"):
    print("Unexpected: file was written despite rejection.")
else:
    print("Confirmed: no file written after rejection.")

## Section 5 - Quality Self-Assessment

The self-assessment runs **regardless of whether you approved or rejected the write**. It evaluates the *search result quality* - not the file write decision.

This cell reads `last_result`, which was set at the end of Section 3 (approve) or Section 4 (reject), whichever you ran most recently. Run both sections to compare how the assessment behaves after each path.

> **Why a standalone LLM call?** The evaluation is intentionally outside the agent graph - simpler to teach, and it runs identically after both paths. In production, you would add this as a post-interrupt node in the graph.

In [ ]:
# Extract the agent's final AI message from the most recent run
msgs_final = last_result["messages"] if isinstance(last_result, dict) else last_result.value['messages']
ai_messages = [m for m in msgs_final if isinstance(m, AIMessage)]
agent_final_message = ai_messages[-1].content if ai_messages else "(no AI message found in result)"

# Build the assessment prompt using string concatenation (avoids triple-quote complexity)
assessment_prompt = (
    f'Assess the following research result for the topic: "{topic}"\n\n'
    f'Result:\n{agent_final_message}\n\n'
    'Evaluate:\n'
    '1. Does the result directly address the topic? (Yes / Partially / No)\n'
    '2. What key aspects of the topic are missing or only partially covered?\n'
    '3. Overall coverage rating: High / Medium / Low - one sentence of justification.'
)

assessment_messages = [{"role": "user", "content": assessment_prompt}]
assessment = llm.invoke(assessment_messages)
display(Markdown("###  Quality Self-Assessment"))
display(Markdown(assessment.content))